# Modèle LASSO — Exploration
**Projet ML 2026 — UMONS | Groupe 3**

LASSO (Least Absolute Shrinkage and Selection Operator) est une régression
linéaire avec pénalité L1 qui force certains coefficients à zéro →
sélection automatique de features.

In [1]:
# Imports et configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import Lasso
from sklearn.model_selection import cross_val_score, KFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

pd.set_option('display.float_format', '{:.4f}'.format)

# Chargement des données
train = pd.read_csv("../data/train_final.csv")
test  = pd.read_csv("../data/test_final.csv")

print(f"✅ Train : {train.shape}")
print(f"✅ Test  : {test.shape}")

✅ Train : (1559, 67)
✅ Test  : (669, 66)


## Préparation des features

In [2]:
# Séparation features / cible
y = train["Ja in Prozent"]

# One Hot Encoding sur Kanton
train_encoded = pd.get_dummies(train, columns=["Kanton"])
test_encoded  = pd.get_dummies(test,  columns=["Kanton"])

# Supprimer les colonnes inutiles
X = train_encoded.drop(columns=["Ja in Prozent", "Gemeinde", "commune_id", "Kantons-Nummer"])
X_test = test_encoded.drop(columns=["Gemeinde", "commune_id", "Kantons-Nummer"])

# Aligner les colonnes train et test
X_test = X_test.reindex(columns=X.columns, fill_value=0)

print(f"X train : {X.shape}")
print(f"X test  : {X_test.shape}")
print(f"y train : {y.shape}")

X train : (1559, 88)
X test  : (669, 88)
y train : (1559,)


## Modèle LASSO de base
On commence par tester différentes valeurs d'alpha pour trouver la meilleure.

In [4]:
# Optimisation de l'hyperparamètre alpha
alphas = [0.0001, 0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
rmse_means = []
rmse_stds = []

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for alpha in alphas:
    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
        ("lasso", Lasso(alpha=alpha, max_iter=10000))
    ])
    scores = cross_val_score(
        pipeline, X, y,
        cv=kf,
        scoring="neg_root_mean_squared_error"
    )
    rmse_means.append(-scores.mean())
    rmse_stds.append(scores.std())

# Meilleur alpha
best_alpha = alphas[np.argmin(rmse_means)]
best_rmse = min(rmse_means)

print("=== Résultats par alpha ===")
for a, r in zip(alphas, rmse_means):
    print(f"alpha={a:.4f} → RMSE CV = {r:.4f}")

print(f"\nMeilleur alpha : {best_alpha}")
print(f"Meilleur RMSE CV : {best_rmse:.4f}")

c:\Users\fling\IdeaProjects\project26ML\venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.916e+04, tolerance: 1.310e+01
  model = cd_fast.enet_coordinate_descent(
c:\Users\fling\IdeaProjects\project26ML\venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.982e+04, tolerance: 1.321e+01
  model = cd_fast.enet_coordinate_descent(
c:\Users\fling\IdeaProjects\project26ML\venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider

=== Résultats par alpha ===
alpha=0.0001 → RMSE CV = 6.6280
alpha=0.0010 → RMSE CV = 6.5629
alpha=0.0100 → RMSE CV = 6.4133
alpha=0.1000 → RMSE CV = 6.3853
alpha=0.5000 → RMSE CV = 6.8257
alpha=1.0000 → RMSE CV = 7.4401
alpha=5.0000 → RMSE CV = 9.4745
alpha=10.0000 → RMSE CV = 10.2601

Meilleur alpha : 0.1
Meilleur RMSE CV : 6.3853
